# RF AUC after Harmony — what is left once a shift between the two sets is allowed
`rf_realism.ipynb` (scGAN protocol, uncorrected PCA) saturates near 1.0 for every comparison across sources, including real vs real from another study
and the same real cells passed through the PBISC read-out. Here the two classes are first aligned with Harmony (batch = class) on the 50 PCs, then the
same random forest (1,000 trees, 5-fold CV) is trained on the corrected PCs. Same cell draws and PCA seeds as the first notebook's lineage-matched repeats 0–1, whose uncorrected AUCs are read back for comparison. Lineage-matched sampling, 3,000 cells per class, 2 repeats.
Reads `rf_realism_cells.h5ad` (the exact cells of the first notebook).

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, harmonypy
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from IPython.display import Image, display
warnings.filterwarnings("ignore"); import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
BASE = os.environ.get("PBISC_BASE", "F:/PBISC"); W = f"{BASE}/bulkonly_lora5_20261006/analysis/integration_report_20261012/rf_realism_20261017"
R, F = f"{W}/results", f"{W}/figs"
plt.rcParams.update({"font.family": ["Arial", "Liberation Sans"], "font.size": 13, "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
A = ad.read_h5ad(f"{W}/rf_realism_cells.h5ad"); print(A); print(A.obs.source.value_counts().to_dict())
SRC = {k: (A[A.obs.source == v].X.tocsr(), A[A.obs.source == v].obs.reset_index(drop=True)) for k, v in
       {"A": "real_A", "B": "real_B", "BT": "real_B_pbisc_readout", "covid": "real_covid", "k1000": "k1000", "k4000": "k4000"}.items()}
COMP = {"C0 real A vs real A": ("A", "A"), "C1 real A vs real B": ("A", "B"), "C2 real A vs real COVID": ("A", "covid"),
        "C3 real A vs real B (PBISC read-out)": ("A", "BT"), "C4 real A vs K = 1,000": ("A", "k1000"), "C5 real A vs K = 4,000": ("A", "k4000"),
        "C6 real B (PBISC read-out) vs K = 1,000": ("BT", "k1000"), "C7 K = 1,000 vs K = 4,000": ("k1000", "k4000")}
LIN = ["T", "NK", "B", "Myeloid", "Other"]; target = SRC["A"][1].lineage.value_counts(normalize=True).reindex(LIN).fillna(0)
def lognorm(X):
    tot = np.asarray(X.sum(1)).ravel(); Y = (sp.diags(1e4 / np.maximum(tot, 1)) @ X).tocsr(); Y.data = np.log1p(Y.data); return Y
def draw(o, n, rng, exclude=None):
    pool = np.arange(len(o)) if exclude is None else np.setdiff1d(np.arange(len(o)), exclude); out = []
    for c in LIN:
        k = int(round(n * target[c])); cand = pool[o.lineage.to_numpy()[pool] == c]
        if k: out.append(rng.choice(cand, min(k, len(cand)), replace=False))
    return np.concatenate(out)
def auc(Z, y, seed):
    a = []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Z, y):
        m = RandomForestClassifier(1000, n_jobs=2, random_state=seed).fit(Z[tr], y[tr]); a.append(roc_auc_score(y[te], m.predict_proba(Z[te])[:, 1]))
    return float(np.mean(a))
rows = []
for name, (s1, s2) in COMP.items():
    for rep in range(2):
        rng = np.random.default_rng(100 + rep); (X1, o1), (X2, o2) = SRC[s1], SRC[s2]
        i1 = draw(o1, 3000, rng); i2 = draw(o2, 3000, rng, exclude=i1 if s1 == s2 else None)
        X = sp.vstack([X1[i1], X2[i2]]).tocsr(); y = np.r_[np.zeros(len(i1)), np.ones(len(i2))]
        P = PCA(50, random_state=rep).fit_transform(lognorm(X).toarray())
        ho = harmonypy.run_harmony(P, pd.DataFrame({"b": y.astype(int).astype(str)}), ["b"], random_state=rep, max_iter_harmony=20, verbose=False)
        Zc = np.asarray(ho.Z_corr); Zc = Zc.T if Zc.shape[0] == 50 else Zc
        rows.append({"comparison": name, "rep": rep, "auc_after_harmony": auc(Zc, y, rep)}); print(name, rep, round(rows[-1]["auc_after_harmony"], 3), flush=True)
U = pd.read_csv(f"{R}/rf_auc_whole.csv"); U = U[(U["mode"] == "matched") & (U.rep < 2)][["comparison", "rep", "auc"]].rename(columns={"auc": "auc_uncorrected"})
H = pd.DataFrame(rows).merge(U, on=["comparison", "rep"]); H.to_csv(f"{R}/rf_auc_after_harmony.csv", index=False)
H.groupby("comparison")[["auc_uncorrected", "auc_after_harmony"]].mean().round(3)

In [ ]:
order = list(COMP); d = H.groupby("comparison")[["auc_uncorrected", "auc_after_harmony"]].agg(["mean", "std"]).reindex(order)
fig, ax = plt.subplots(figsize=(13, 6.2)); y = np.arange(len(order))[::-1]
for dy, k, col, lab in [(0.18, "auc_uncorrected", "#9AA3B2", "uncorrected PCA (scGAN protocol)"), (-0.18, "auc_after_harmony", "#2A78D6", "after Harmony (batch = class)")]:
    ax.barh(y + dy, d[(k, "mean")] - 0.5, left=0.5, height=0.34, color=col, label=lab, xerr=d[(k, "std")], error_kw={"lw": 1})
    for yi, v in zip(y + dy, d[(k, "mean")]): ax.text(v + 0.01, yi, f"{v:.2f}", va="center", fontsize=11)
ax.set_yticks(y, [c.split(" ", 1)[1] for c in order], fontsize=12); ax.set_xlim(0.45, 1.06); ax.axvline(0.5, color="#1F2430", lw=1)
ax.axvline(0.65, color="#C4373A", lw=1, ls="--"); ax.text(0.655, len(order) - 0.45, "scGAN vs real: 0.65", color="#C4373A", fontsize=11)
ax.set_xlabel("random forest AUC (5-fold CV), lineage-matched, 3,000 cells per class"); ax.legend(loc="lower right", fontsize=11)
plt.tight_layout(); fig.savefig(f"{F}/rf_auc_after_harmony.png"); plt.close(fig); display(Image(f"{F}/rf_auc_after_harmony.png", width=1000))